In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Concatenate, Embedding, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Set seed agar hasil konsisten
np.random.seed(42)
tf.random.set_seed(42)

print("=" * 60)
print("🚀 Smart Queue Model Training")
print("=" * 60)

# ========== DATA LOADING ==========
print("\n📊 Loading data...")
csv_path = "data/dataset_RS2_final.csv"

if not os.path.exists(csv_path):
    print(f"❌ File not found: {csv_path}")
    print("Pastikan file CSV ada di lokasi yang benar!")
    exit(1)

df = pd.read_csv(csv_path)
print(f"✅ Data loaded: {df.shape[0]} rows × {df.shape[1]} columns")

🚀 Smart Queue Model Training

📊 Loading data...
✅ Data loaded: 20000 rows × 24 columns


In [8]:
# ========== FEATURE ENGINEERING ==========
print("\n🔧 Feature engineering...")
df['waktu_kedatangan'] = pd.to_datetime(df['waktu_kedatangan'])
df['waktu_registrasi'] = pd.to_datetime(df['waktu_registrasi'])
df['durasi_registrasi_menit'] = (df['waktu_registrasi'] - df['waktu_kedatangan']).dt.total_seconds() / 60

# ========== DATA CLEANING ==========
print("🧹 Removing data leakage & redundant columns...")
kolom_buang = [
    'waktu_mulai_layanan', 'waktu_selesai_layanan', 'durasi_layanan',
    'kepuasan_pasien', 'calc_wait', 'biaya', 'tanggal',
    'waktu_kedatangan', 'waktu_registrasi', 'waktu_tunggu',
    'asuransi_enc', 'prioritas_enc', 'status_pasien_enc', 'nama_poli_enc'
]

X = df.drop(columns=kolom_buang)
y = df['waktu_tunggu'].values

# Pisahkan fitur
num_cols = ['umur', 'jumlah_antrian', 'jam_kedatangan', 'is_peak', 'durasi_registrasi_menit']
cat_cols = ['jenis_kelamin', 'asuransi', 'status_pasien', 'prioritas', 'nama_poli', 'hari']

print(f"✅ Features: {len(num_cols)} numerical + {len(cat_cols)} categorical")


🔧 Feature engineering...
🧹 Removing data leakage & redundant columns...
✅ Features: 5 numerical + 6 categorical


In [9]:
# ========== DATA PREPROCESSING ==========
print("\n⚙️  Preprocessing data...")

# Scaling numerik
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[num_cols])
X_num_df = pd.DataFrame(X_num_scaled, columns=num_cols)

# Encoding kategorikal
encoder = OrdinalEncoder()
X_cat_encoded = encoder.fit_transform(X[cat_cols])
X_cat_df = pd.DataFrame(X_cat_encoded, columns=cat_cols)

# Gabung
X_final = pd.concat([X_num_df, X_cat_df], axis=1)
cat_uniques = {col: len(np.unique(X_final[col])) for col in cat_cols}

print(f"✅ Data ready for training")

# Split data: 80% train, 10% val, 10% test
X_temp, X_test, y_temp, y_test = train_test_split(X_final, y, test_size=0.10, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1111, random_state=42)

def df_to_dict(df_features, num_c, cat_c):
    ds_dict = {col: df_features[col].values for col in cat_c}
    ds_dict['numeric_inputs'] = df_features[num_c].values
    return ds_dict

train_dict = df_to_dict(X_train, num_cols, cat_cols)
val_dict = df_to_dict(X_val, num_cols, cat_cols)
test_dict = df_to_dict(X_test, num_cols, cat_cols)

print(f"📦 Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")


⚙️  Preprocessing data...
✅ Data ready for training
📦 Train: 16000 | Val: 2000 | Test: 2000


In [10]:
# ========== BUILD MODEL ==========
print("\n🧠 Building neural network...")

inputs = []
embedding_outputs = []

# Categorical inputs with embeddings
for col in cat_cols:
    input_cat = Input(shape=(1,), name=col)
    inputs.append(input_cat)

    vocab_size = cat_uniques[col]
    embed_dim = min(50, max(2, int(vocab_size ** 0.5)))

    embed_layer = Embedding(input_dim=vocab_size, output_dim=embed_dim)(input_cat)
    flatten_layer = Flatten()(embed_layer)
    embedding_outputs.append(flatten_layer)

# Numeric inputs
input_num = Input(shape=(len(num_cols),), name='numeric_inputs')
inputs.append(input_num)

# Concatenate all features
concat = Concatenate()(embedding_outputs + [input_num])

# Hidden layers
x = Dense(128, activation='relu')(concat)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.1)(x)

x = Dense(32, activation='relu')(x)

# Output layer
output = Dense(1, activation='linear')(x)

# Compile
model = Model(inputs=inputs, outputs=output)
model.compile(optimizer=Adam(learning_rate=0.005), loss='mse', metrics=['mae'])

print(f"✅ Model built successfully")
model.summary()



🧠 Building neural network...
✅ Model built successfully


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ jenis_kelamin       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ asuransi            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ status_pasien       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prioritas           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nama_poli           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hari (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 2)      │          4 │ jenis_kelamin[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 2)      │          4 │ asuransi[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 2)      │          4 │ status_pasien[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 1, 2)      │          4 │ prioritas[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 1, 2)      │         12 │ nama_poli[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 1, 2)      │         14 │ hari[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 2)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 2)         │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 2)         │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 2)         │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 2)         │          0 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 2)         │          0 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_inputs      │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                 

 Total params: 13,483 (52.67 KB)

 Trainable params: 13,099 (51.17 KB)

 Non-trainable params: 384 (1.50 KB)

In [11]:
# ========== TRAIN MODEL ==========
print("\n📈 Training model...")

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.0001, verbose=0)

history = model.fit(
    x=train_dict,
    y=y_train,
    validation_data=(val_dict, y_val),
    epochs=100,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)



📈 Training model...
Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 149.2701 - mae: 8.2511 - val_loss: 161.0780 - val_mae: 10.5034 - learning_rate: 0.0050
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 24.7497 - mae: 3.7761 - val_loss: 50.1205 - val_mae: 5.3748 - learning_rate: 0.0050
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 22.9988 - mae: 3.6053 - val_loss: 17.2497 - val_mae: 3.2414 - learning_rate: 0.0050
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 20.9237 - mae: 3.4309 - val_loss: 15.1022 - val_mae: 2.9318 - learning_rate: 0.0050
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 20.1113 - mae: 3.3396 - val_loss: 14.5159 - val_mae: 2.7547 - learning_rate: 0.0050
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 18.9927 - mae: 3.2583 - val_loss: 17.6058 - val_mae: 2.9618 - learning_rate: 0.0050
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 18.3814 - mae: 3.2170 - val_loss: 15.5400 - v

In [12]:
# ========== EVALUATION ==========
print("\n✅ Evaluating on test set...")
test_loss, test_mae = model.evaluate(test_dict, y_test, verbose=0)
print(f"Test MAE: {test_mae:.2f} minutes")

y_pred = model.predict(test_dict, verbose=0).flatten()

print("\n" + "=" * 60)
print("📊 Sample Predictions (5 examples):")
print("=" * 60)
for i in range(5):
    print(f"Actual: {y_test[i]:.1f} min → Predicted: {y_pred[i]:.1f} min | Error: {abs(y_test[i] - y_pred[i]):.1f} min")


✅ Evaluating on test set...
Test MAE: 2.62 minutes

📊 Sample Predictions (5 examples):
Actual: 0.0 min → Predicted: 0.4 min | Error: 0.4 min
Actual: 30.0 min → Predicted: 37.0 min | Error: 7.0 min
Actual: 26.0 min → Predicted: 21.2 min | Error: 4.8 min
Actual: 19.0 min → Predicted: 21.2 min | Error: 2.2 min
Actual: 46.0 min → Predicted: 50.3 min | Error: 4.3 min


In [13]:
# ========== SAVE MODEL & PREPROCESSING ==========
print("\n💾 Saving model files...")

model_dir = os.getcwd()
save_path = os.path.join(model_dir, "model_artifacts")

# Buat folder kalau belum ada
os.makedirs(save_path, exist_ok=True)

# Simpan semua ke folder yang sama
model.save(os.path.join(save_path, 'smart_queue_model.keras'))
joblib.dump(scaler, os.path.join(save_path, 'scaler.joblib'))
joblib.dump(encoder, os.path.join(save_path, 'encoder.joblib'))

metadata = {
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'version': '1.0',
    'test_mae': float(test_mae),
    'model_type': 'tensorflow_keras'
}
joblib.dump(metadata, os.path.join(save_path, 'metadata.joblib'))

print(f"✅ Models saved to {save_path}/")
print(f"   - smart_queue_model.keras")
print(f"   - scaler.joblib")
print(f"   - encoder.joblib")
print(f"   - metadata.joblib")

print("\n" + "=" * 60)
print(f"🎉 TRAINING COMPLETE!")
print(f"Model accuracy (MAE): {test_mae:.2f} minutes")
print("=" * 60)


💾 Saving model files...
✅ Models saved to /Users/leaves/Sandbox/Capstone/ml/model_artifacts/
   - smart_queue_model.keras
   - scaler.joblib
   - encoder.joblib
   - metadata.joblib

🎉 TRAINING COMPLETE!
Model accuracy (MAE): 2.62 minutes
